In [0]:
import json
import re

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType, DecimalType, IntegerType, LongType,
    StringType, StructField, StructType,
    TimestampNTZType, TimestampType
)

BASE = "/Volumes/workspace/bronze_core/qa_snapshot"
spark.conf.set("spark.sql.session.timeZone", "UTC")

with open(f"{BASE}/manifest.json", encoding="utf-8") as file:
    manifest = json.load(file)


def converter(nome, coluna):
    valor = F.col(f"`{nome}`")
    valor = F.when(F.trim(valor) == "", F.lit(None)).otherwise(valor)
    tipo = coluna["data_type"].lower()

    if tipo == "boolean":
        texto = F.lower(valor)
        return (
            F.when(texto.isin("t", "true"), F.lit(True))
             .when(texto.isin("f", "false"), F.lit(False))
             .otherwise(valor.cast("boolean"))
        )
    if tipo == "integer":
        return valor.cast(IntegerType())
    if tipo == "bigint":
        return valor.cast(LongType())
    if tipo == "numeric":
        precisao = coluna["numeric_precision"]
        escala = coluna["numeric_scale"]
        if precisao is not None and escala is not None and precisao <= 38:
            return valor.cast(DecimalType(precisao, escala))
        return valor  # Precisão indefinida: preserva o texto original.
    if tipo == "date":
        return valor.cast(DateType())
    if tipo == "timestamp with time zone":
        return valor.cast(TimestampType())
    if tipo == "timestamp without time zone":
        return valor.cast(TimestampNTZType())

    return valor  # UUID, texto, enum e array permanecem como texto.


for tabela, info in manifest["tables"].items():
    if not re.fullmatch(r"[a-z][a-z0-9_]*", tabela):
        raise ValueError(f"Nome de tabela inválido: {tabela}")

    colunas = info["columns"]
    esquema_csv = StructType([
        StructField(coluna["name"], StringType(), True)
        for coluna in colunas
    ])

    destino = f"workspace.bronze_core.{tabela}"
    if spark.catalog.tableExists(destino):
        raise RuntimeError(
            f"{destino} já existe. Pare e confira antes de substituir qualquer dado."
        )

    dados = (
        spark.read
        .option("header", "true")
        .option("enforceSchema", "false")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "FAILFAST")
        .schema(esquema_csv)
        .csv(f"{BASE}/{tabela}.csv")
    )

    dados = dados.select(*[
        converter(coluna["name"], coluna).alias(coluna["name"])
        for coluna in colunas
    ])

    quantidade = dados.count()
    if quantidade != info["rows"]:
        raise RuntimeError(
            f"{tabela}: esperado {info['rows']} linhas, encontrado {quantidade}"
        )

    dados.writeTo(destino).using("delta").create()
    print(f"{destino}: {quantidade} linhas")